# 03 — PaliGemma2-3B (mix) + LoRA


## 1. Установка зависимостей

In [ ]:
!pip install -q "pillow<12"
!pip install -q -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 44.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 54.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 35.8 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [ ]:
import torch, gc, json, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PRED_DIR = "predictions"
os.makedirs(PRED_DIR, exist_ok=True)


CUDA available: True
Tesla T4


## 2. Данные VK (deepvk)

- `deepvk/LLaVA-Instruct-ru` — обучающие пары (изображение + инструкция + ответ на русском).
- `deepvk/GQA-ru` — тест визуального рассуждения, метрика ExactMatch (короткий ответ).
- `deepvk/MMBench-ru` — тест с вариантами ответа A/B/C/D.


In [ ]:
from datasets import load_dataset

gqa_images = load_dataset("deepvk/GQA-ru", "testdev_balanced_images", split="testdev")
gqa_instructions = load_dataset("deepvk/GQA-ru", "testdev_balanced_instructions", split="testdev")
gqa_images_by_id = {ex["id"]: ex["image"] for ex in gqa_images}
gqa_ru = gqa_instructions.shuffle(seed=42)
print(gqa_ru)
print(gqa_ru[0])


README.md:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

testdev_balanced_images/testdev-00000-of(…):   0%|          | 0.00/65.6M [00:00<?, ?B/s]

Generating testdev split:   0%|          | 0/398 [00:00<?, ? examples/s]

testdev_balanced_instructions/testdev-00(…):   0%|          | 0.00/1.95M [00:00<?, ?B/s]

Generating testdev split:   0%|          | 0/12216 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'imageId', 'question', 'answer', 'fullAnswer', 'isBalanced', 'groups', 'entailed', 'equivalent', 'types', 'annotations', 'semantic', 'semanticStr'],
    num_rows: 12216
})
{'id': '202060197', 'imageId': 'n14087', 'question': 'За каким предметом мебели находятся шторы?', 'answer': 'диван', 'fullAnswer': 'Предмет мебели - это диван.', 'isBalanced': True, 'groups': {'global': 'furniture', 'local': '15-curtains_behind,o'}, 'entailed': "['202060025', '202060024', '202060198']", 'equivalent': "['202060197']", 'types': {'structural': 'query', 'semantic': 'rel', 'detailed': 'categoryRelO'}, 'annotations': {'question': [{'objectId': '8', 'value': '20'}, {'objectId': '3:6', 'value': '13'}], 'answer': [{'objectId': '0', 'value': '13'}], 'fullAnswer': [{'objectId': '1:4', 'value': '13'}, {'objectId': '6', 'value': '13'}]}, 'semantic': [{'operation': 'select', 'argument': 'curtains (20)', 'dependencies': []}, {'operation': 'relate', 'argument': 'furniture,behind,o (13

In [ ]:
mmbench_ru = load_dataset("deepvk/MMBench-ru", split="dev").shuffle(seed=42)
print(mmbench_ru)
print(mmbench_ru[0])


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

mmbench_ru_dev.parquet:   0%|          | 0.00/92.7M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/3910 [00:00<?, ? examples/s]

Dataset({
    features: ['index', 'question', 'hint', 'A', 'B', 'C', 'D', 'answer', 'category', 'image', 'source', 'l2-category', 'comment', 'split'],
    num_rows: 3910
})
{'index': 926, 'question': 'Какое из следующих утверждений соответствует картинке?', 'hint': 'nan', 'A': 'Серый круг находится слева от циановой фигуры.', 'B': 'Циановый квадрат находится слева от серого круга.', 'C': 'Циановый эллипс находится справа от серого круга.', 'D': 'Циановый круг находится справа от круга.', 'answer': 'A', 'category': 'attribute_comparison', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=100x100 at 0x7C4E266AFDD0>, 'source': 'shapeworld', 'l2-category': 'finegrained_perception (cross-instance)', 'comment': 'world-71.png', 'split': 'dev'}


In [ ]:
train_ds = load_dataset("deepvk/LLaVA-Instruct-ru", split="train")
print(train_ds)
print(train_ds[0])


README.md:   0%|          | 0.00/2.67k [00:00<?, ?B/s]

llava_instruct_ru_train.json:   0%|          | 0.00/197M [00:00<?, ?B/s]

llava_instruct_ru_val.json:   0%|          | 0.00/77.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/109905 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/34075 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations', 'type', 'id', 'image'],
    num_rows: 109905
})
{'conversations': [{'from': 'human', 'value': '<image>\nЧто делает обстановку этой ванны такой особенной по сравнению с другими?'}, {'from': 'gpt', 'value': 'Эта ванная комната выделяется своей чистотой и опрятностью. На изображении видно, что ванная идеально чистая, свет включен. Наличие цветов, стоящих на столешнице, добавляет элемент уюта и свежести в интерьер. Деревянные шкафчики придают теплоту и натуральность обстановке. Зелено-белый полосатый душевой занавес создает яркий акцент и придает привлекательность дизайну ванной комнаты.'}], 'type': 'complex_reasoning', 'id': '000000253464', 'image': 'coco/train2017/000000253464.jpg'}


### 2.1 Скачиваем нужные изображения COCO для обучающей подвыборки


In [ ]:
import os, requests
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

TRAIN_SUBSET_SIZE = 2000
COCO_CACHE_DIR = "coco_cache"
COCO_DOWNLOAD_WORKERS = 32
os.makedirs(COCO_CACHE_DIR, exist_ok=True)

def _download_one(rel_path):
    key = rel_path.split("coco/", 1)[-1]
    local_path = os.path.join(COCO_CACHE_DIR, key)
    if not os.path.exists(local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        url = "http://images.cocodataset.org/" + key
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(resp.content)
    return local_path

def load_coco_image(rel_path):
    return Image.open(_download_one(rel_path)).convert("RGB")

train_subset = train_ds.select(range(min(TRAIN_SUBSET_SIZE, len(train_ds))))
paths = [ex["image"] for ex in train_subset]

failed = []
with ThreadPoolExecutor(max_workers=COCO_DOWNLOAD_WORKERS) as pool:
    futures = {pool.submit(_download_one, p): p for p in paths}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Скачиваю картинки COCO"):
        try:
            future.result()
        except Exception as e:
            failed.append((futures[future], str(e)))

print(f"Готово, картинки в {COCO_CACHE_DIR}/. Не удалось скачать: {len(failed)}")
if failed[:5]:
    print("Примеры ошибок:", failed[:5])


Скачиваю картинки COCO:   0%|          | 0/2000 [00:00<?, ?it/s]

Готово, картинки в coco_cache/. Не удалось скачать: 1
Примеры ошибок: [('coco/train2017/000000072790.jpg', '503 Server Error: Service Unavailable for url: http://images.cocodataset.org/train2017/000000072790.jpg')]


## 3. Общие утилиты

In [ ]:
import re

EVAL_N = 200
NUM_EPOCHS = 1

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower()).strip(".,!?\"'")

GQA_SUFFIX = " Ответь одним словом."

def extract_qa(example, images_by_id):
    image = images_by_id.get(example.get("imageId"))
    question = example.get("question")
    answer = example.get("answer")
    if question is not None:
        question = question + GQA_SUFFIX
    return image, question, answer

def extract_mmbench(example):
    image = example.get("image")
    question = example.get("question")
    options = []
    for letter in ["A", "B", "C", "D"]:
        opt = example.get(letter)
        if opt:
            options.append(f"{letter}. {opt}")
    full_question = question + "\n" + "\n".join(options) + "\nОтветь только буквой правильного варианта."
    gold = str(example.get("answer")).strip().upper()
    return image, full_question, gold

def save_predictions(records, path):
    with open(os.path.join(PRED_DIR, path), "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def evaluate_gqa(generate_fn, dataset, images_by_id, n, tag):
    correct, total, records = 0, 0, []
    pbar = tqdm(range(min(n, len(dataset))), desc=f"GQA-ru [{tag}]")
    for i in pbar:
        image, question, gold = extract_qa(dataset[i], images_by_id)
        if image is None or question is None or gold is None:
            continue
        pred = generate_fn(image, question)
        is_correct = normalize(pred) == normalize(gold)
        correct += int(is_correct)
        total += 1
        records.append({"idx": i, "gold": gold, "pred": pred, "correct": is_correct})
        pbar.set_postfix(acc=f"{correct/total:.3f}" if total else "—")
    acc = correct / total if total else 0.0
    print(f"[{tag}] GQA-ru ExactMatch: {acc:.4f} ({correct}/{total})")
    save_predictions(records, f"gqa_{tag}.json")
    return acc

def evaluate_mmbench(generate_fn, dataset, n, tag):
    correct, total, records = 0, 0, []
    pbar = tqdm(range(min(n, len(dataset))), desc=f"MMBench-ru [{tag}]")
    for i in pbar:
        image, question, gold = extract_mmbench(dataset[i])
        if image is None or gold is None:
            continue
        pred_text = generate_fn(image, question, max_new_tokens=4)
        pred_letter = "".join(c for c in pred_text.upper() if c in "ABCD")[:1]
        is_correct = pred_letter == gold
        correct += int(is_correct)
        total += 1
        records.append({"idx": i, "gold": gold, "pred": pred_letter, "correct": is_correct})
        pbar.set_postfix(acc=f"{correct/total:.3f}" if total else "—")
    acc = correct / total if total else 0.0
    print(f"[{tag}] MMBench-ru accuracy: {acc:.4f} ({correct}/{total})")
    save_predictions(records, f"mmbench_{tag}.json")
    return acc


## 4. PaliGemma2-3B (mix) + LoRA

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

PALI_MODEL_ID = "google/paligemma2-3b-mix-224"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

pali_processor = AutoProcessor.from_pretrained(PALI_MODEL_ID)
pali_model = PaliGemmaForConditionalGeneration.from_pretrained(
    PALI_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

In [ ]:
@torch.no_grad()
def pali_generate(image, question, max_new_tokens=16):
    inputs = pali_processor(text=question, images=image, return_tensors="pt").to(pali_model.device)
    input_len = inputs["input_ids"].shape[-1]
    out = pali_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_ids = out[:, input_len:]
    return pali_processor.decode(gen_ids[0], skip_special_tokens=True)

### 4.1 Baseline

In [ ]:
pali_baseline_gqa = evaluate_gqa(pali_generate, gqa_ru, gqa_images_by_id, EVAL_N, "pali_baseline")
pali_baseline_mmbench = evaluate_mmbench(pali_generate, mmbench_ru, EVAL_N, "pali_baseline")


GQA-ru [pali_baseline]:   0%|          | 0/200 [00:00<?, ?it/s]

[pali_baseline] GQA-ru ExactMatch: 0.1400 (28/200)


MMBench-ru [pali_baseline]:   0%|          | 0/200 [00:00<?, ?it/s]

[pali_baseline] MMBench-ru accuracy: 0.5350 (107/200)


### 4.2 LoRA-дообучение

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

pali_model = prepare_model_for_kbit_training(pali_model)

pali_lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # ПРОВЕРИТЬ: имена проекций в Gemma2-декодере
    task_type="CAUSAL_LM",
)
pali_model = get_peft_model(pali_model, pali_lora_config)
pali_model.print_trainable_parameters()


trainable params: 9,375,744 || all params: 3,041,618,160 || trainable%: 0.3082


In [ ]:
def build_training_example_pali(example):
    conv = example.get("conversations")
    if conv:
        human_turn = next((t["value"] for t in conv if t.get("from") == "human"), "")
        gpt_turn = next((t["value"] for t in conv if t.get("from") == "gpt"), "")
    else:
        human_turn = example.get("question", "")
        gpt_turn = example.get("answer", "")
    return {"image": load_coco_image(example["image"]), "prompt": human_turn, "answer": gpt_turn}

def pali_collate_fn(batch):
    images = [b["image"] for b in batch]
    prompts = [b["prompt"] for b in batch]
    full_texts = [b["prompt"] + " " + b["answer"] for b in batch]

    full_inputs = pali_processor(text=full_texts, images=images, return_tensors="pt", padding=True)
    prompt_inputs = pali_processor(text=prompts, images=images, return_tensors="pt", padding=True)

    labels = full_inputs["input_ids"].clone()
    labels[labels == pali_processor.tokenizer.pad_token_id] = -100
    for i, prompt_ids in enumerate(prompt_inputs["input_ids"]):
        prompt_len = int((prompt_ids != pali_processor.tokenizer.pad_token_id).sum())
        labels[i, :prompt_len] = -100

    full_inputs["labels"] = labels
    return full_inputs

In [ ]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

GRAD_ACCUM_STEPS = 2

pali_train_subset = train_subset
pali_processed_train = [build_training_example_pali(ex) for ex in pali_train_subset]
pali_train_loader = DataLoader(pali_processed_train, batch_size=1, shuffle=True, collate_fn=pali_collate_fn)

pali_optimizer = torch.optim.AdamW(pali_model.parameters(), lr=1e-4)
pali_num_steps = NUM_EPOCHS * len(pali_train_loader) // GRAD_ACCUM_STEPS
pali_scheduler = get_cosine_schedule_with_warmup(pali_optimizer, num_warmup_steps=20, num_training_steps=pali_num_steps)

pali_model.train()
pbar = tqdm(total=pali_num_steps, desc="PaliGemma2-3B LoRA")
running_loss, opt_step = 0.0, 0
for epoch in range(NUM_EPOCHS):
    for i, batch in enumerate(pali_train_loader):
        batch = {k: v.to(pali_model.device) for k, v in batch.items()}
        loss = pali_model(**batch).loss / GRAD_ACCUM_STEPS
        loss.backward()
        if (i + 1) % GRAD_ACCUM_STEPS == 0:
            pali_optimizer.step()
            pali_scheduler.step()
            pali_optimizer.zero_grad()
            opt_step += 1
            running_loss += loss.item() * GRAD_ACCUM_STEPS
            pbar.update(1)
            pbar.set_postfix(
                loss=f"{loss.item()*GRAD_ACCUM_STEPS:.4f}",
                avg_loss=f"{running_loss/opt_step:.4f}",
                lr=f"{pali_scheduler.get_last_lr()[0]:.2e}",
            )
pbar.close()

pali_model.save_pretrained("paligemma2-3b-ru-lora")
pali_processor.save_pretrained("paligemma2-3b-ru-lora")

PaliGemma2-3B LoRA:   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


['paligemma2-3b-ru-lora/processor_config.json']

### 4.3 Оценка после LoRA-дообучения

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*special image tokens.*")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

In [ ]:
pali_model.eval()
pali_finetuned_gqa = evaluate_gqa(pali_generate, gqa_ru, gqa_images_by_id, EVAL_N, "pali_finetuned")
pali_finetuned_mmbench = evaluate_mmbench(pali_generate, mmbench_ru, EVAL_N, "pali_finetuned")

print(f"GQA-ru:     baseline={pali_baseline_gqa:.4f}  finetuned={pali_finetuned_gqa:.4f}")
print(f"MMBench-ru: baseline={pali_baseline_mmbench:.4f}  finetuned={pali_finetuned_mmbench:.4f}")


GQA-ru [pali_finetuned]:   0%|          | 0/200 [00:00<?, ?it/s]

[pali_finetuned] GQA-ru ExactMatch: 0.0850 (17/200)


MMBench-ru [pali_finetuned]:   0%|          | 0/200 [00:00<?, ?it/s]

[pali_finetuned] MMBench-ru accuracy: 0.2350 (47/200)
GQA-ru:     baseline=0.1400  finetuned=0.0850
MMBench-ru: baseline=0.5350  finetuned=0.2350
